# Step10a — Mapas Estáticos / Atlas (Ecuador, 2015–2025)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 10 — Productos, sección A: atlas de 8 mapas estáticos (Fase 0 §9)
**Autor / Author:** Jerson Achicanoy, Information Management and GIS Expert
**Fecha:** 2026-08-06

**Objetivo de este notebook:** producir el "dataset maestro" cantonal que
alimenta el resto de la Fase 10 (mapas interactivos, dashboard, perfiles),
y los 8 mapas temáticos estáticos pedidos en el diseño original
(Fase 0 §9 "Maps"): Frecuencia, Severidad, Frecuencia ENSO, Población
Afectada, Pérdidas Agrícolas, Pérdidas de Vivienda, Cronicidad (HCI),
Hotspots — en PNG individuales y como un atlas PDF consolidado.

**decisión confirmada por el equipo (AskUserQuestion, 2026-08-06):**
formato **atlas estático + interactivo** (este notebook cubre la parte
estática; `Step10b_Ecuador_Interactive_Maps.ipynb` cubre la interactiva).

**Nivel de agregación:** cantón (221 unidades) — el nivel más detallado
disponible en todo el proyecto desde la Fase 2, consistente con los
rankings/perfiles del resto de la Fase 10.

**English:** Static maps / atlas for Ecuador, 2015–2025. Builds the master canton dataset that feeds the rest of the product steps (interactive maps, dashboard, profiles) and the static thematic maps: event frequency, severity, ENSO frequency, affected population, agricultural losses, housing losses, chronicity index and hotspots, as individual PNGs and as one consolidated PDF atlas. Variants exist per hazard type (flood, debris flow, landslide, scour).

## 0. Configuración del entorno

**English:** Environment setup: imports, configuration, logger and output folders.

In [ ]:
# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper that creates folders when missing
from src.utilidades.io_utils import ensure_dir

# Tabular data
import pandas as pd
# Numerical arrays
import numpy as np
# Vector geodata
import geopandas as gpd
# Plotting library
import matplotlib.pyplot as plt
# Colour utilities
import matplotlib.colors as mcolors
# Multi-page PDF writer for the atlas
from matplotlib.backends.backend_pdf import PdfPages
# Classification schemes (Natural Breaks / Jenks)
import mapclassify

# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration
config = load_config(CONFIG_PATH)
# Create the notebook logger
logger = get_logger("step10a_static_maps", config, log_dir=PROJECT_ROOT / "logs")
# Log the start of the notebook
logger.info("Fase 10a iniciada — configuración cargada desde %s", CONFIG_PATH)

# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Output folder for the PNG maps
maps_dir = ensure_dir(PROJECT_ROOT / "outputs" / "maps")
# Output folder for figures
figures_dir = ensure_dir(PROJECT_ROOT / "outputs" / "figures")

# Default font size for all figures
plt.rcParams["font.size"] = 9

## 1. Dataset maestro cantonal

Une, en una sola tabla espacial, todas las métricas ya calculadas en
fases anteriores que se van a mapear — no se recalcula ningún índice,
solo se reutilizan las salidas ya validadas de las Fases 3-9. Se exporta
una vez (`mapa_master_canton.gpkg`) para que el resto de la Fase 10
(mapas interactivos, dashboard, perfiles) lo reutilice sin repetir esta
unión.

**English:** Master canton dataset: joins in one spatial table every metric already computed in earlier steps that will be mapped. No index is recomputed; validated outputs are reused. It is exported once so the remaining product notebooks can read it.

In [ ]:
# Read canton polygons
canton = gpd.read_file(processed_dir / "adm2_canton.geojson")
# Canton codes as strings
canton["canton_code"] = canton["canton_code"].astype(str)

# --- HCI + 4 sub-índices (Fase 8) ---
# Read the composite index at canton level
hci = pd.read_csv(processed_dir / "hci_composite_canton.csv", dtype={"canton_code": str})
hci_todos = hci[hci["hazard_type"] == "TODOS"][
    ["canton_code", "severity_index", "recurrence_index", "persistence_index",
     "enso_magnitude_mean", "hci", "clase_hci"]
]

# --- Frecuencia de eventos (Fase 3) ---
# Read event recurrence per canton
freq = pd.read_csv(processed_dir / "hist_recurrencia_canton.csv", dtype={"canton_code": str})
freq_todos = freq[freq["hazard_type"] == "TODOS"][["canton_code", "n_eventos", "anios_afectados"]] \
    .rename(columns={"n_eventos": "frecuencia_inundaciones"})

# --- Frecuencia ENSO (Fase 3, mismo criterio "evento ENSO" de Fase 5 §4.3: ICEN, prefijo "El Niño") ---
# Read event counts per ENSO category
enso = pd.read_csv(processed_dir / "hist_enso_canton.csv", dtype={"canton_code": str})
# Keep ICEN 'El Niño' months only (same event definition as the recurrence step)
enso_icen = enso[
    (enso["hazard_type"] == "TODOS") & (enso["indice_enso"] == "ICEN")
    & (enso["categoria"].str.startswith("El Niño"))
]
# Sum El Niño events per canton
enso_freq = enso_icen.groupby("canton_code")["n_eventos"].sum().reset_index() \
    .rename(columns={"n_eventos": "frecuencia_enso"})

# --- Hotspots Gi* (Fase 9) ---
# Read Getis-Ord Gi* hotspot classes
hotspot = pd.read_csv(processed_dir / "hotspots_gi_star_canton.csv", dtype={"canton_code": str})
# Keep canton code and Gi* class
hotspot_hci = hotspot[["canton_code", "gi_clase_hci"]]

# --- Impactos máximos históricos: población / agricultura / vivienda ---
# Recalculados directamente de hazard_event_geo (máximo histórico por cantón,
# misma filosofía de "peor escenario esperable" que Severidad Fase 4 §4.1),
# no se reutiliza hist_impactos_canton porque viene en formato largo
# (una variable a la vez) y aquí se necesitan combinadas (afectado+destruido).
# Read all georeferenced events
he = gpd.read_file(processed_dir / "hazard_event_geo.gpkg", layer="hazard_event")
# Canton codes as strings
he["canton_code"] = he["canton_code"].astype(str)
# Population affected = direct + indirect
he["poblacion_total"] = he["affected_direct"].fillna(0) + he["affected_indirect"].fillna(0)
# Crops lost = affected + destroyed hectares
he["agricultura_total"] = he["crops_affected_ha"].fillna(0) + he["crops_destroyed_ha"].fillna(0)
# Housing lost = affected + destroyed houses
he["vivienda_total"] = he["houses_affected"].fillna(0) + he["houses_destroyed"].fillna(0)
# --- Educación y Salud (añadido 2026-09-18): mismo patrón "afectado+destruido",
# máximo histórico por cantón -- confirman datos ya presentes en hazard_event_geo
# pero que hasta ahora no tenían mapa propio (solo alimentaban el pilar
# "Infraestructura" del Índice de Severidad, Fase 4). Ver
# docs/methodology/10_products.md §9 para el detalle de esta adición.
# Schools affected + destroyed
he["educacion_total"] = he["schools_affected"].fillna(0) + he["schools_destroyed"].fillna(0)
# Health facilities affected + destroyed
he["salud_total"] = he["health_facilities_affected"].fillna(0) + he["health_facilities_destroyed"].fillna(0)
# Historical maximum of each impact per canton
impact_max = he.groupby("canton_code")[
    ["poblacion_total", "agricultura_total", "vivienda_total", "educacion_total", "salud_total"]
].max().reset_index()

# --- Conectividad y bienes (añadido 2026-09-18, Fase 14): variables presentes en la
# base cruda (BD_2015_2025) desde el inicio del proyecto pero nunca incorporadas --
# ni al Índice de Severidad ni a ningún producto. Ver docs/methodology/14_variables_complementarias.md.
# Se extraen con un join propio por event_id (no viven en hazard_event_geo) y se
# agregan con la misma filosofía "máximo histórico por cantón" del resto de esta celda.
# Read complementary variables (roads, bridges, assets)
extra = pd.read_csv(processed_dir / "variables_complementarias_canton.csv", dtype={"canton_code": str})
# Complementary variables to map
extra_cols = ["vias_metros_afectados", "puentes_total", "bienes_publicos_total", "bienes_privados_total",
              "familias_afectadas", "familias_damnificadas", "damnificados_total", "personas_acogientes"]

# Master canton table: geometry + all metrics to map
master = canton.merge(hci_todos, on="canton_code", how="left") \
    .merge(freq_todos, on="canton_code", how="left") \
    .merge(enso_freq, on="canton_code", how="left") \
    .merge(hotspot_hci, on="canton_code", how="left") \
    .merge(impact_max, on="canton_code", how="left") \
    .merge(extra, on="canton_code", how="left")
# Fill missing complementary variables with 0
master[extra_cols] = master[extra_cols].fillna(0)  # cantones sin ese tipo de impacto -> 0, no nulo

master["frecuencia_enso"] = master["frecuencia_enso"].fillna(0)  # cantones sin eventos en meses ENSO -> 0, no nulo

cols_no_geom = [c for c in master.columns if c != "geometry"]
assert master[cols_no_geom].isna().sum().sum() == 0, "quedaron valores nulos en el dataset maestro — revisar uniones"
assert len(master) == 221, f"se esperaban 221 cantones, hay {len(master)}"

# Export the master dataset as GeoPackage
master.to_file(processed_dir / "mapa_master_canton.gpkg", driver="GPKG")
# Export the master dataset as CSV
master.drop(columns="geometry").to_csv(processed_dir / "mapa_master_canton.csv", index=False)
logger.info("Dataset maestro cantonal: %d cantones x %d columnas, exportado", len(master), len(master.columns))
master.head(3)

## 2. Función genérica de mapa coroplético

Una sola función parametrizable para las 6 variables continuas (usa
quiebres naturales/Jenks, 5 clases — misma convención de clasificación
que la Fase 8), evita repetir la misma lógica de graficado 6 veces.
Los mapas categóricos (HCI ya clasificado, Hotspots) usan una función
aparte (Sección 4) porque no son variables continuas a re-clasificar.

**English:** Generic choropleth function: one parametrised function for the continuous variables (Natural Breaks / Jenks, 5 classes, same convention as the composite index) to avoid repeating the plotting logic. Categorical maps (classified chronicity, hotspots) are drawn separately.

In [ ]:
# Province boundaries drawn over each map
PROVINCE_BOUNDARY = gpd.read_file(processed_dir / "adm1_province.geojson")

# Generic choropleth map function (Natural Breaks, 5 classes)
def mapa_coropletico(gdf, columna, titulo, archivo, cmap="YlOrRd", k=5, fmt="{:,.0f}"):
    """Genera y guarda un mapa coroplético cantonal con quiebres naturales (Jenks).

    Args:
        gdf: GeoDataFrame maestro cantonal (con geometría y la columna a mapear).
        columna: nombre de la columna numérica a clasificar y colorear.
        titulo: título del mapa.
        archivo: nombre de archivo (sin extensión) para guardar en outputs/maps.
        cmap: mapa de color de matplotlib.
        k: número de clases (Jenks).
        fmt: formato de número para la leyenda.

    Returns:
        La figura de matplotlib generada (para reutilizar en el atlas PDF).
    """
    # New figure, portrait format
    fig, ax = plt.subplots(figsize=(8, 10))
    # Draw the cantons coloured by class
    gdf.plot(
        column=columna, ax=ax, cmap=cmap, scheme="NaturalBreaks", k=k,
        legend=True, edgecolor="white", linewidth=0.2,
        legend_kwds={"title": titulo, "fmt": fmt, "loc": "lower left", "fontsize": 7, "title_fontsize": 8},
        missing_kwds={"color": "lightgrey"},
    )
    # Overlay province outlines
    PROVINCE_BOUNDARY.boundary.plot(ax=ax, color="black", linewidth=0.4)
    # Map title
    ax.set_title(titulo, fontsize=13, fontweight="bold")
    # Hide the axes
    ax.set_axis_off()
    # Small footer text
    ax.annotate(
        "HCI Ecuador · 2015-2025 · Inundación + Deslizamiento + Socavamiento + Aluvión", xy=(0.99, 0.01), xycoords="axes fraction",
        fontsize=6, color="gray", ha="right",
    )
    # Fit the layout
    fig.tight_layout()
    # Save the PNG
    fig.savefig(maps_dir / f"{archivo}.png", dpi=150, bbox_inches="tight")
    # Log the generated map
    logger.info("Mapa estático generado: %s.png", archivo)
    # Return the figure for the PDF atlas
    return fig

## 3. Los 10 mapas continuos (Jenks, 5 clases)

**English:** The continuous maps (Jenks, 5 classes).

In [ ]:
# Continuous maps: (column, title, file name, colour map)
MAPAS_CONTINUOS = [
    ("frecuencia_inundaciones", "Frecuencia de Eventos (Nº de eventos, 2015-2025)", "01_frecuencia_inundaciones", "Blues"),
    ("severity_index", "Índice de Severidad", "02_severidad", "OrRd"),
    ("frecuencia_enso", "Frecuencia de eventos en meses El Niño", "03_frecuencia_enso", "PuBu"),
    ("poblacion_total", "Población Afectada (máximo histórico, directa+indirecta)", "04_poblacion_afectada", "YlOrRd"),
    ("agricultura_total", "Pérdidas Agrícolas (máximo histórico, ha afectadas+destruidas)", "05_perdidas_agricolas", "YlGn"),
    ("vivienda_total", "Pérdidas de Vivienda (máximo histórico, viviendas afectadas+destruidas)", "06_perdidas_vivienda", "Oranges"),
    ("educacion_total", "Establecimientos Educativos Afectados (máximo histórico, afectados+destruidos)", "09_educacion_afectada", "BuPu"),
    ("salud_total", "Centros de Salud Afectados (máximo histórico, afectados+destruidos)", "10_salud_afectada", "RdPu"),
    ("vias_metros_afectados", "Vías Afectadas (máximo histórico, metros lineales)", "11_vias_afectadas", "Greys"),
    ("puentes_total", "Puentes Afectados (máximo histórico, afectados+destruidos)", "12_puentes_afectados", "PuRd"),
]

# Figures kept for the atlas
figuras_atlas = []
# Draw one map per variable
for columna, titulo, archivo, cmap in MAPAS_CONTINUOS:
    fig = mapa_coropletico(master, columna, titulo, archivo, cmap=cmap)
    figuras_atlas.append(fig)
    plt.close(fig)  # libera memoria, ya se guardó el PNG y la referencia para el atlas

## 4. Mapa de Cronicidad HCI (clases ya fijadas en Fase 8, no se recalculan)

**English:** Chronicity index map: for hazard-specific notebooks the Jenks classes are recomputed on the hazard's own distribution; in the combined notebook the classes come from the composite index step.

In [ ]:
# Class names from the configuration
CLASS_LABELS = config["hci_composite"]["class_labels"]  # ["Muy Baja", ..., "Muy Alta"]
# Sequential red palette, 5 classes
CLASS_COLORS = ["#fee5d9", "#fcae91", "#fb6a4a", "#de2d26", "#a50f15"]  # secuencial rojo, 5 clases
# Map each class to its colour
color_map_hci = dict(zip(CLASS_LABELS, CLASS_COLORS))

fig, ax = plt.subplots(figsize=(8, 10))
# Draw one class at a time
for clase, color in color_map_hci.items():
    sub = master[master["clase_hci"] == clase]
    if len(sub):
        sub.plot(ax=ax, color=color, edgecolor="white", linewidth=0.2, label=clase)
PROVINCE_BOUNDARY.boundary.plot(ax=ax, color="black", linewidth=0.4)
ax.set_title("Cronicidad por El Niño (HCI)", fontsize=12, fontweight="bold")
ax.set_axis_off()
# Legend handles
handles = [plt.Rectangle((0, 0), 1, 1, color=color_map_hci[c]) for c in CLASS_LABELS]
# Legend
ax.legend(handles, CLASS_LABELS, loc="lower left", fontsize=8, title="Clase HCI", title_fontsize=9)
ax.annotate("HCI Ecuador · 2015-2025 · Inundación + Deslizamiento + Socavamiento + Aluvión", xy=(0.99, 0.01), xycoords="axes fraction", fontsize=6, color="gray", ha="right")
fig.tight_layout()
fig.savefig(maps_dir / "07_hci_cronicidad.png", dpi=150, bbox_inches="tight")
logger.info("Mapa estático generado: 07_hci_cronicidad.png")
figuras_atlas.append(fig)
plt.close(fig)

## 5. Mapa de Hotspots (Getis-Ord Gi*, Fase 9)

**English:** Hotspot map (Getis-Ord Gi*): statistically significant clusters of high (hot) and low (cold) chronicity.

In [ ]:
# Hotspot classes from hot to cold
HOTSPOT_ORDER = [
    "Punto Caliente 99%", "Punto Caliente 95%", "Punto Caliente 90%",
    "No significativo",
    "Punto Frío 90%", "Punto Frío 95%", "Punto Frío 99%",
]
# Diverging palette: red = hot, blue = cold
HOTSPOT_COLORS = ["#67000d", "#cb181d", "#fc9272", "#f0f0f0", "#9ecae1", "#3182bd", "#08306b"]
# Map each hotspot class to its colour
color_map_hotspot = dict(zip(HOTSPOT_ORDER, HOTSPOT_COLORS))

fig, ax = plt.subplots(figsize=(8, 10))
# Draw one class at a time
for clase, color in color_map_hotspot.items():
    sub = master[master["gi_clase_hci"] == clase]
    if len(sub):
        sub.plot(ax=ax, color=color, edgecolor="white", linewidth=0.2)
PROVINCE_BOUNDARY.boundary.plot(ax=ax, color="black", linewidth=0.4)
ax.set_title("Hotspots de Cronicidad HCI (Getis-Ord Gi*)", fontsize=12, fontweight="bold")
ax.set_axis_off()
handles = [plt.Rectangle((0, 0), 1, 1, color=color_map_hotspot[c]) for c in HOTSPOT_ORDER]
ax.legend(handles, HOTSPOT_ORDER, loc="lower left", fontsize=7, title="Clasificación Gi*", title_fontsize=8)
ax.annotate("HCI Ecuador · 2015-2025 · Inundación + Deslizamiento + Socavamiento + Aluvión", xy=(0.99, 0.01), xycoords="axes fraction", fontsize=6, color="gray", ha="right")
fig.tight_layout()
fig.savefig(maps_dir / "08_hotspots_gi_star.png", dpi=150, bbox_inches="tight")
logger.info("Mapa estático generado: 08_hotspots_gi_star.png")
figuras_atlas.append(fig)
plt.close(fig)

## 6. Atlas PDF consolidado (los 10 mapas en un solo documento)

**English:** Consolidated PDF atlas: all maps in one document, with a cover page.

In [ ]:
# Output path of the consolidated atlas PDF
atlas_path = PROJECT_ROOT / "outputs" / "reports" / "Atlas_HCI_Ecuador.pdf"
# Create the reports folder
ensure_dir(atlas_path.parent)

# Regenera las figuras en memoria (se habían cerrado tras guardarlas como PNG,
# más simple y robusto que mantener 8 figuras abiertas en memoria a la vez).
# Se listan por nombre explícito (no glob("*.png")) porque outputs/maps/ también
# contiene, desde la Fase 11, los mapas SAR (mapa_sar_flood_susceptibility*.png)
# -- un glob amplio los incluiría por error en el atlas de 8 mapas de la Fase 10.
# Explicit list of map files (not a glob, other steps also write maps)
nombres_atlas = [c[2] for c in MAPAS_CONTINUOS] + ["07_hci_cronicidad", "08_hotspots_gi_star"]
# Full paths of the PNG files
png_files = [maps_dir / f"{n}.png" for n in nombres_atlas]
# Check every PNG exists
assert all(p.exists() for p in png_files), "falta algún PNG del atlas"
# Check the expected number of maps
assert len(png_files) == 12, f"se esperaban 12 mapas PNG, hay {len(png_files)}"

# Write the atlas, one map per page
with PdfPages(atlas_path) as pdf:
    # Cover page
    # Portada
    fig, ax = plt.subplots(figsize=(8.5, 11))
    ax.axis("off")
    ax.text(0.5, 0.65, "Atlas de Cronicidad de\nInundaciones por El Niño", ha="center", fontsize=22, fontweight="bold")
    ax.text(0.5, 0.55, "Ecuador 2015–2025", ha="center", fontsize=16)
    ax.text(0.5, 0.45, "HCI — Hazard Chronicity Index", ha="center", fontsize=12, style="italic")
    ax.text(0.5, 0.15, "Jerson Achicanoy · 2026-08-06", ha="center", fontsize=10)
    # Save the page
    pdf.savefig(fig)
    plt.close(fig)

    # Add each map as a page
    for png in png_files:
        # Read the PNG
        img = plt.imread(png)
        fig, ax = plt.subplots(figsize=(8.5, 11))
        # Show the image on the page
        ax.imshow(img)
        ax.axis("off")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

logger.info("Atlas PDF consolidado: %s (%d páginas)", atlas_path, len(png_files) + 1)
# Report the atlas path
print(f"Atlas exportado: {atlas_path}")

## 7. Validación

- Los 8 PNG existen en `outputs/maps/`.
- El PDF del atlas tiene 9 páginas (portada + 8 mapas).
- El dataset maestro (`mapa_master_canton.gpkg`) no tiene nulos y cubre
  los 221 cantones — verificado en la Sección 1 antes de exportar.

**English:** Validation: all PNG files exist, the atlas has the expected number of pages and the master dataset has no nulls and covers all 221 cantons.

In [ ]:
assert all((maps_dir / f"{n}.png").exists() for n in nombres_atlas), "falta algun PNG del atlas de Fase 10"
# Check the atlas was written
assert atlas_path.exists() and atlas_path.stat().st_size > 0, "el atlas PDF no se genero correctamente"
logger.info("Fase 10a - validacion OK. Mapas en %s, atlas en %s", maps_dir, atlas_path)
print("Fase 10a completa.")